# Deep Learning Unit Test Automation (Aufgabe 2) — MNIST Version

This Colab notebook demonstrates automated testing and logging for a deep learning model using the MNIST dataset. It includes the model, logging/timing decorators, training, prediction, and unittests for fit() and predict(). No external data files are required.

# Deep Learning Unit Test Automation (Aufgabe 2)

This Colab notebook demonstrates automated testing and logging for a deep learning model using the Bank Note dataset. It includes the model, logging/timing decorators, training, prediction, and unittests for fit() and predict().

## 1. Setup: Install and Import Libraries

In [ ]:
!pip install scikit-learn tensorflow keras pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow import keras
import time
import logging

## 3. Load MNIST Data

We use the MNIST dataset, which is built into Keras. No file upload is required.

In [ ]:
from tensorflow.keras.datasets import mnist

# Load MNIST data
def load_mnist():
    (X_train, y_train), (X_test, y_test) = mnist.load_data()
    # Normalize and flatten
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    X_train = X_train.reshape(-1, 28*28)
    X_test = X_test.reshape(-1, 28*28)
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_mnist()

## 3. Load Data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Preprocessing

In [ ]:
def create_model():
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(784,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Logging and timing decorators (must be defined before use)
import time
import functools

def my_logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished.")
        return result
    return wrapper

def my_timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f} seconds.")
        return result, elapsed
    return wrapper

## 5. Model Definition

In [ ]:
@my_logger
@my_timer
def fit_model(model, X, y, epochs=5, batch_size=128):
    history = model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)
    return history

@my_logger
def predict_model(model, X):
    preds = np.argmax(model.predict(X), axis=1)
    return preds

## 7. Unit Test 1: Test predict() on MNIST Test Data

In [ ]:
@my_logger
@my_timer
def fit_model(model, X, y, epochs=10, batch_size=8):
    history = model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)
    return history

@my_logger
def predict_model(model, X):
    preds = (model.predict(X) > 0.5).astype('int32')
    return preds

## 7. Unit Test 1: Test predict() on Test Data

In [ ]:
# Train model
model = create_model()
_, reference_time = fit_model(model, X_train_scaled, y_train, epochs=5, batch_size=128)

# Test predict
preds = predict_model(model, X_test_scaled)
acc = accuracy_score(y_test, preds)
cm = confusion_matrix(y_test, preds)
print(f'\nAccuracy: {acc:.2f}')
print(f'Confusion Matrix:\n{cm}')

## 8. Unit Test 2: Test fit() Runtime

In [ ]:
# Test fit() runtime
_, elapsed = fit_model(model, X_train_scaled, y_train, epochs=5, batch_size=128)
print(f'\nReference fit time: {reference_time:.4f}s, Current fit time: {elapsed:.4f}s')
assert elapsed <= 1.2 * reference_time, 'fit() runtime exceeded 120% of reference time'

## 9. Conclusion

- Both unit tests (predict and fit) are executed in this notebook.
- Logging and timing outputs are shown above.
- You can upload your own test data or adjust parameters as needed.
- All steps are reproducible in Google Colab.

## 10. Unittest Implementation for Colab

The following cell implements the required unittests for `fit()` and `predict()` using Python's `unittest` framework. The fit test checks that the runtime does not exceed 120% of a reference, and the predict test checks accuracy and confusion matrix. All outputs are visible below.

In [ ]:
import unittest

class TestDeepLearningModel(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.model = create_model()
        _, cls.reference_time = fit_model(cls.model, X_train_scaled, y_train, epochs=5, batch_size=128)

    def test_predict_accuracy(self):
        preds = predict_model(self.model, X_test_scaled)
        acc = accuracy_score(y_test, preds)
        cm = confusion_matrix(y_test, preds)
        print(f"\nAccuracy: {acc:.2f}")
        print(f"Confusion Matrix:\n{cm}")
        self.assertGreater(acc, 0.9, "Accuracy is too low!")

    def test_fit_time(self):
        _, elapsed = fit_model(self.model, X_train_scaled, y_train, epochs=5, batch_size=128)
        print(f"\nReference fit time: {self.reference_time:.4f}s, Current fit time: {elapsed:.4f}s")
        self.assertLessEqual(elapsed, 1.2 * self.reference_time, "fit() runtime exceeded 120% of reference time!")

unittest.main(argv=[''], verbosity=2, exit=False)